# Electricity Load Forecasting with Google TimesFM (PJM)

## Industry Problem

Electricity providers must forecast demand accurately to avoid:

- **Under-forecasting** -> blackout risk and emergency balancing costs.
- **Over-forecasting** -> unnecessary generator startups and fuel waste.

This notebook builds a production-style forecasting workflow with **Google TimesFM** for:

- **Next hour** (`H=1`)
- **Next day** (`H=24`)
- **Next week** (`H=168`)

on real hourly grid load data from PJM (US).

## What You Will Learn

1. How to acquire and quality-check hourly electricity load data.
2. How to frame utility forecasting horizons in one unified pipeline.
3. How to run TimesFM with calendar covariates (XReg mode).
4. How to benchmark against strong naive baselines.
5. How to translate forecasts into operational decisions (reserve and cost risk).

In [ ]:
from __future__ import annotations

import math
import os
import subprocess
import zipfile
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Keep the run deterministic where possible.
np.random.seed(42)

# Paths
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'data' / 'electricity_load'
RAW_DIR = DATA_DIR / 'raw'
ART_DIR = PROJECT_ROOT / 'artifacts' / 'electricity_load_timesfm'

RAW_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = RAW_DIR / 'PJM_Load_hourly.csv'

print('Project root:', PROJECT_ROOT)
print('CSV path:', CSV_PATH)

## 1) Data Download (Web)

Primary source: Kaggle dataset `robikscube/hourly-energy-consumption`.

The cell below is robust:

- Uses local cached file if present.
- Otherwise tries Kaggle CLI download.
- Fails with a clear message if download credentials/network are unavailable.

In [ ]:
def ensure_dataset(csv_path: Path) -> Path:
    if csv_path.exists():
        print(f'Found existing file: {csv_path}')
        return csv_path

    zip_path = RAW_DIR / 'hourly-energy-consumption.zip'

    cmd = [
        'kaggle',
        'datasets',
        'download',
        '-d',
        'robikscube/hourly-energy-consumption',
        '-p',
        str(RAW_DIR),
        '--force',
    ]

    try:
        print('Downloading dataset from Kaggle...')
        subprocess.run(cmd, check=True, capture_output=True, text=True)
    except Exception as exc:
        raise RuntimeError(
            'Failed to download dataset from Kaggle. Ensure Kaggle API is configured or place '
            f'PJM_Load_hourly.csv at {csv_path}.'
        ) from exc

    if zip_path.exists():
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(RAW_DIR)

    if not csv_path.exists():
        raise FileNotFoundError(f'Expected file not found after download: {csv_path}')

    return csv_path


csv_path = ensure_dataset(CSV_PATH)
csv_path

## 2) Load + Data Quality Checks

In [ ]:
raw_df = pd.read_csv(csv_path, parse_dates=['Datetime'])
raw_df = raw_df.rename(columns={'Datetime': 'timestamp', 'PJM_Load_MW': 'load_mw'})
raw_df = raw_df.sort_values('timestamp')

print('Raw shape:', raw_df.shape)
print('Date range:', raw_df['timestamp'].min(), '->', raw_df['timestamp'].max())
print('Null rows:', raw_df['load_mw'].isna().sum())
print('Duplicate timestamps:', raw_df['timestamp'].duplicated().sum())

# Regularize to strict hourly grid and fill small/large gaps by interpolation.
df = raw_df.drop_duplicates('timestamp').set_index('timestamp').asfreq('h')
missing_after_grid = int(df['load_mw'].isna().sum())
df['load_mw'] = df['load_mw'].interpolate(limit_direction='both')

print('Rows after hourly regularization:', len(df))
print('Missing values introduced by grid alignment:', missing_after_grid)
print('Any missing after interpolation:', bool(df['load_mw'].isna().any()))

df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df.index, df['load_mw'], lw=0.6)
ax.set_title('PJM Hourly Load (MW)')
ax.set_ylabel('MW')
ax.set_xlabel('Time')
plt.tight_layout()
plt.show()

## 3) Forecasting Configuration

We will evaluate TimesFM on utility-relevant horizons:

- `1` hour ahead (real-time balancing)
- `24` hours ahead (day-ahead commitment)
- `168` hours ahead (week-ahead planning)

Backtest strategy: 4 weekly anchors near the end of history.

In [ ]:
@dataclass
class Config:
    context_len: int = 1024
    max_horizon: int = 168
    eval_horizons: tuple[int, ...] = (1, 24, 168)
    anchors_weeks_ago: tuple[int, ...] = (4, 3, 2, 1)
    per_core_batch_size: int = 8
    xreg_mode: str = 'xreg + timesfm'
    xreg_ridge: float = 1e-3


cfg = Config()

last_pos = len(df) - 1
anchor_positions: list[int] = []
for weeks_ago in cfg.anchors_weeks_ago:
    end_pos = last_pos - weeks_ago * 168
    if end_pos - cfg.context_len + 1 >= 0 and end_pos + cfg.max_horizon < len(df):
        anchor_positions.append(end_pos)

print('Anchors selected:', len(anchor_positions))
print('Anchor timestamps:')
for pos in anchor_positions:
    print(' -', df.index[pos])

## 4) Load TimesFM and Build Forecast Helpers

In [ ]:
# Optional: force CPU behavior consistency in mixed environments.
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '')

import timesfm

model = timesfm.TimesFM_2p5_200M_torch.from_pretrained('google/timesfm-2.5-200m-pytorch')
forecast_config = timesfm.ForecastConfig(
    max_context=cfg.context_len,
    max_horizon=cfg.max_horizon,
    normalize_inputs=True,
    per_core_batch_size=cfg.per_core_batch_size,
    use_continuous_quantile_head=True,
    force_flip_invariance=True,
    infer_is_positive=True,
    fix_quantile_crossing=True,
    return_backcast=True,
)
model.compile(forecast_config)
print('TimesFM compiled.')

In [ ]:
def build_dynamic_covariates(start_ts: pd.Timestamp, context_len: int, horizon: int) -> dict[str, list[list[float]]]:
    full_dates = pd.date_range(start_ts, periods=context_len + horizon, freq='h')
    hour = full_dates.hour.values.astype(np.float32)
    dow = full_dates.dayofweek.values.astype(np.float32)

    return {
        'hour_sin': [np.sin(2 * np.pi * hour / 24.0).astype(np.float32).tolist()],
        'hour_cos': [np.cos(2 * np.pi * hour / 24.0).astype(np.float32).tolist()],
        'dow_sin': [np.sin(2 * np.pi * dow / 7.0).astype(np.float32).tolist()],
        'dow_cos': [np.cos(2 * np.pi * dow / 7.0).astype(np.float32).tolist()],
        'is_weekend': [(dow >= 5).astype(np.float32).tolist()],
    }


def run_timesfm_forecast(context: np.ndarray, start_ts: pd.Timestamp, horizon: int) -> tuple[np.ndarray, np.ndarray]:
    cov = build_dynamic_covariates(start_ts, len(context), horizon)

    try:
        point, quant = model.forecast_with_covariates(
            inputs=[context.astype(np.float32)],
            dynamic_numerical_covariates=cov,
            xreg_mode=cfg.xreg_mode,
            ridge=cfg.xreg_ridge,
        )
    except Exception:
        point, quant = model.forecast(horizon=horizon, inputs=[context.astype(np.float32)])

    point_np = np.asarray(point, dtype=np.float32)[0, :horizon]
    quant_np = np.asarray(quant, dtype=np.float32)[0, :horizon, :]

    point_np = np.clip(point_np, 0.0, None)
    quant_np = np.clip(quant_np, 0.0, None)
    return point_np, quant_np

## 5) Rolling Backtest: TimesFM vs Baselines

In [ ]:
def wmape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.abs(y_true - y_pred).sum() / (np.abs(y_true).sum() + 1e-8))


records: list[dict] = []
weekly_predictions = []

for anchor_pos in anchor_positions:
    context = df['load_mw'].iloc[anchor_pos - cfg.context_len + 1 : anchor_pos + 1].to_numpy(np.float32)
    future = df['load_mw'].iloc[anchor_pos + 1 : anchor_pos + cfg.max_horizon + 1].to_numpy(np.float32)

    anchor_ts = df.index[anchor_pos]
    start_ts = df.index[anchor_pos - cfg.context_len + 1]

    tfm_pred, tfm_quant = run_timesfm_forecast(context=context, start_ts=start_ts, horizon=cfg.max_horizon)

    naive_last = np.repeat(context[-1], cfg.max_horizon)
    seasonal24 = np.tile(context[-24:], math.ceil(cfg.max_horizon / 24))[: cfg.max_horizon]

    models = {
        'timesfm': tfm_pred,
        'naive_last': naive_last,
        'seasonal24': seasonal24,
    }

    for hz in cfg.eval_horizons:
        y = future[:hz]
        for name, pred in models.items():
            p = pred[:hz]
            records.append(
                {
                    'anchor_ts': anchor_ts,
                    'horizon': hz,
                    'model': name,
                    'mae': mean_absolute_error(y, p),
                    'rmse': mean_squared_error(y, p) ** 0.5,
                    'wmape': wmape(y, p),
                }
            )

    # Save one concrete weekly forecast from the latest anchor for operations.
    if anchor_pos == max(anchor_positions):
        forecast_index = pd.date_range(anchor_ts + pd.Timedelta(hours=1), periods=cfg.max_horizon, freq='h')
        weekly_predictions = pd.DataFrame(
            {
                'timestamp': forecast_index,
                'timesfm_p50_mw': tfm_pred,
                'timesfm_q10_mw': tfm_quant[:, 1],
                'timesfm_q90_mw': tfm_quant[:, 9],
            }
        )

metrics = (
    pd.DataFrame(records)
    .groupby(['horizon', 'model'], as_index=False)[['mae', 'rmse', 'wmape']]
    .mean()
    .sort_values(['horizon', 'rmse'])
)

metrics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=False)
for ax, metric in zip(axes, ['mae', 'rmse', 'wmape']):
    pivot = metrics.pivot(index='horizon', columns='model', values=metric)
    pivot.plot(kind='bar', ax=ax)
    ax.set_title(metric.upper())
    ax.set_xlabel('Horizon (hours)')
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

## 6) Utility Operations Layer

Convert the 168-hour forecast into operational recommendations:

- **Reserve margin** from uncertainty (`q90 - q50`)
- **Recommended committed supply** = `p50 + reserve`
- **Cost-risk simulation**:
  - Under-forecast penalty: high (blackout/emergency balancing risk)
  - Over-forecast penalty: lower (fuel/commitment inefficiency)

In [ ]:
# Build operational schedule from latest forecast.
reserve_floor_mw = 250.0
weekly_predictions['reserve_mw'] = np.maximum(
    weekly_predictions['timesfm_q90_mw'] - weekly_predictions['timesfm_p50_mw'],
    reserve_floor_mw,
)
weekly_predictions['recommended_commit_mw'] = weekly_predictions['timesfm_p50_mw'] + weekly_predictions['reserve_mw']

# Prepare latest actual week for quick risk simulation.
latest_anchor = max(anchor_positions)
actual_week = df['load_mw'].iloc[latest_anchor + 1 : latest_anchor + cfg.max_horizon + 1].to_numpy(np.float32)

p50 = weekly_predictions['timesfm_p50_mw'].to_numpy(np.float32)
seasonal24 = np.tile(
    df['load_mw'].iloc[latest_anchor - 23 : latest_anchor + 1].to_numpy(np.float32),
    math.ceil(cfg.max_horizon / 24),
)[: cfg.max_horizon]

under_cost_per_mwh = 180.0
over_cost_per_mwh = 35.0


def imbalance_cost(actual: np.ndarray, forecast: np.ndarray) -> float:
    err = forecast - actual
    over = np.clip(err, 0.0, None).sum() * over_cost_per_mwh
    under = np.clip(-err, 0.0, None).sum() * under_cost_per_mwh
    return float(over + under)

cost_timesfm = imbalance_cost(actual_week, p50)
cost_seasonal24 = imbalance_cost(actual_week, seasonal24)

print(f'Imbalance cost (TimesFM):   ${cost_timesfm:,.0f}')
print(f'Imbalance cost (Seasonal24): ${cost_seasonal24:,.0f}')
if cost_seasonal24 > 0:
    print(f'Estimated cost reduction:    {(cost_seasonal24 - cost_timesfm) / cost_seasonal24:.1%}')

In [ ]:
weekly_predictions[['timestamp', 'timesfm_p50_mw', 'timesfm_q10_mw', 'timesfm_q90_mw', 'reserve_mw', 'recommended_commit_mw']].head(30)

In [ ]:
# Save artifacts for downstream grid-planning workflows.
metrics_path = ART_DIR / 'backtest_metrics.csv'
ops_path = ART_DIR / 'weekly_operational_schedule.csv'

metrics.to_csv(metrics_path, index=False)
weekly_predictions.to_csv(ops_path, index=False)

print('Saved:', metrics_path)
print('Saved:', ops_path)

## 7) Final Answers to the Business Forecast Requests

Using the latest anchor in history:

- **Next hour forecast**: first row of `weekly_predictions`.
- **Next day forecast**: first 24 rows.
- **Next week forecast**: full 168 rows.

The saved operational schedule includes uncertainty-aware reserve recommendations to reduce blackout risk while controlling fuel/startup cost.